# Cursed Tomb — Persistent Deck Evolution & Solvability Analysis

This notebook analyzes how persistent card degradation (scars, curses, entombment) impacts deck composition and empirical solvability across rounds in *The Cursed Tomb*.

**Key features:**
- Pure core simulation reuse (`sim.deck_evolution_core`) without requiring CLI arguments or `multiprocessing.Pool`.
- Kernel-persistent `runs` state allowing additive overlays across different configurations.
- Interactive widget controls (if `ipywidgets` is installed) or static fallback cells.
- Fast CSV loading path to regenerate plots from heavy CLI simulation sweeps.

In [ ]:
import sys
import os
import random
from pathlib import Path

# Ensure repository root is in Python path
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

%matplotlib inline
import matplotlib.pyplot as plt

from sim.deck_evolution_core import (
    run_collapse_campaign,
    aggregate_results,
    plot_evolution,
    write_aggregated_csv,
    load_aggregated_csv,
    RuleFlags,
    DIFFICULTIES,
    HAS_MPL,
)

In [ ]:
# Initialize persistent runs list if not already present in kernel environment
if 'runs' not in globals():
    runs = []

def render_plots():
    """Render the 3-panel figure for all current runs in memory."""
    if not runs:
        print("No runs currently stored in memory. Add a run using widgets or code below.")
        return
    plot_evolution(runs, show=True)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    difficulty_w = widgets.Dropdown(
        options=list(DIFFICULTIES.keys()),
        value='archaeologist',
        description='Difficulty:'
    )
    solver_w = widgets.Dropdown(
        options=['greedy', 'heuristic', 'beam', 'dfs'],
        value='greedy',
        description='Solver:'
    )
    campaigns_w = widgets.IntSlider(
        value=10, min=1, max=100, step=5,
        description='Campaigns:'
    )
    max_rounds_w = widgets.IntSlider(
        value=30, min=5, max=100, step=5,
        description='Max Rounds:'
    )
    probes_w = widgets.IntSlider(
        value=10, min=1, max=100, step=5,
        description='Probes:'
    )

    btn_add = widgets.Button(description="Add Run", button_style="success", icon="plus")
    btn_clear = widgets.Button(description="Clear All Runs", button_style="danger", icon="trash")
    out_area = widgets.Output()

    def on_add_clicked(b):
        with out_area:
            clear_output(wait=True)
            diff = difficulty_w.value
            sol = solver_w.value
            c_count = campaigns_w.value
            mr = max_rounds_w.value
            pr = probes_w.value

            print(f"Running simulation: {diff}/{sol} ({c_count} campaigns, max {mr} rounds)...")
            flags = RuleFlags(
                scars=True, curses=True, blessings=True, attrition=True,
                sealed_tomb_victory=False, rank_anchor_victory=False
            )
            max_redeals = DIFFICULTIES[diff]
            rng = random.Random(random.randint(0, 1_000_000_000))

            campaign_results = []
            for _ in range(c_count):
                res = run_collapse_campaign(
                    rng=rng,
                    max_redeals=max_redeals,
                    flags=flags,
                    max_rounds=mr,
                    solver_name=sol,
                    probe_solver_name='greedy',
                    n_probes=pr,
                    sample_interval=1,
                )
                campaign_results.append(res)

            agg, summary = aggregate_results(campaign_results, mr, 1)
            run_label = f"{diff}/{sol} #{len(runs)+1}"
            runs.append({"label": run_label, "data": agg})
            print(f"Added run '{run_label}'. Total active runs: {len(runs)}")
            render_plots()

    def on_clear_clicked(b):
        global runs
        with out_area:
            clear_output(wait=True)
            runs = []
            plt.close('all')
            print("Cleared all stored runs from memory.")

    btn_add.on_click(on_add_clicked)
    btn_clear.on_click(on_clear_clicked)

    display(widgets.VBox([
        difficulty_w,
        solver_w,
        campaigns_w,
        max_rounds_w,
        probes_w,
        widgets.HBox([btn_add, btn_clear]),
        out_area
    ]))
except ImportError:
    print("ipywidgets not installed. Use programmatic cells below.")

In [ ]:
# Run a campaign configuration programmatically
flags = RuleFlags(
    scars=True, curses=True, blessings=True, attrition=True,
    sealed_tomb_victory=False, rank_anchor_victory=False
)
rng = random.Random(42)

results = [
    run_collapse_campaign(
        rng=rng,
        max_redeals=DIFFICULTIES['archaeologist'],
        flags=flags,
        max_rounds=30,
        solver_name='greedy',
        probe_solver_name='greedy',
        n_probes=10,
        sample_interval=1,
    )
    for _ in range(10)
]

agg, summary = aggregate_results(results, max_rounds=30, sample_interval=1)

# Append to persistent runs if empty
if not runs:
    runs.append({"label": "archaeologist/greedy #1", "data": agg})

render_plots()

In [ ]:
# To load a pre-computed CSV from a CLI simulation sweep:
csv_file = "/tmp/agg.csv"  # Replace with path to your CSV
if os.path.exists(csv_file):
    loaded_agg = load_aggregated_csv(csv_file)
    runs.append({"label": f"CLI CSV ({os.path.basename(csv_file)})", "data": loaded_agg})
    render_plots()
else:
    print(f"CSV file not found at '{csv_file}'. Run CLI with --csv first to generate it.")

In [ ]:
# Reset persistent state manually
runs = []
plt.close('all')
print("State reset complete.")